# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Setup - make sure we're in the right directory
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv")
print("Starter data found. You're ready!")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready!


## 1. My lane (or freestyle) and why



Lane 4: Search Intent (predefined lane from the lane guide).

Why this lane: the dataset has three content types and four intent types, so intent-content alignment is measurable here rather than assumed. In the starter CSV, roughly a quarter to a third of pages sit in combinations that do not obviously match (navigational intent on a keyword article), and decline rates differ across intent buckets and the output, a ranked queue of pages for a content editor to review, maps cleanly to a real decision.  Secondary motivation is from a security angle (flagging unusual intent-content pairings), not the main claim.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Dataset: {df.shape[0]:,} pages x {df.shape[1]} columns")
print(f"Declining pages (observed): {df['is_declining_label'].mean():.1%}")
print()
print("Intent x content_type counts:")
print(df.groupby(["main_intent", "content_type"]).size().unstack(fill_value=0).to_string())
print()
# How many pages sit in a combination that is rare (<0.5% of the dataset)?
combo = df.groupby(["main_intent", "content_type"]).size().reset_index(name="n")
combo["pct"] = combo["n"] / len(df) * 100
rare = combo[combo["pct"] < 0.5]
print(f"Rare intent-content combinations (<0.5% each): {len(rare)}")
print(rare.to_string(index=False))

Dataset: 30,000 pages x 45 columns
Declining pages (observed): 54.2%

Intent x content_type counts:
content_type   keyword article  comparison article
main_intent                                       
commercial                4612                   0
informational            16538                 697
navigational                46                   0
transactional             5733                   0

Rare intent-content combinations (<0.5% each): 1
 main_intent    content_type  n      pct
navigational keyword article 46 0.153333


## 2. The question: decision, action, cost of a wrong call

Research question: For pages that already have search visibility, does intent-content alignment predict which pages are more likely to be declining and can we rank pages by that risk so a human reviews the most likely mismatches first?

The decision this improves:which pages a content editor opens first for an intent-alignment review.

Who acts, and what they do:
- Content team: rewrite or re-categorize the page so the content matches the searcher's intent.
- SEO team: adjust targeting (title, meta, internal links) when the content is fine but the intent signal is off.


Cost of a wrong call:
- False positive (flagged as a risk but actually fine): 2–4 editor hours wasted per page.
- False negative (a genuinely misaligned page not flagged): the page keeps declining, and the mismatch compounds over time.

Why data/ML helps at all: intent-content alignment is a small, discrete signal on its own, but combined with visibility, CTR-vs-position, and page age it becomes a many-signal, messy prioritization problem, exactly the shape where a hand-written rule runs out and a ranked score earns its place.

In [4]:
# Framing the decision with real numbers
visible = df[df["impressions_90d"] >= 500].copy()

# How many visible pages would a human realistically need to sort?
n_visible = len(visible)
n_declining_visible = int(visible["is_declining_label"].sum())
print(f"Visible pages (>=500 impressions_90d): {n_visible:,}")
print(f"Of those, declining: {n_declining_visible:,} ({visible['is_declining_label'].mean():.1%})")
print()
print("A content team going through ~50 pages/week would need "
      f"{n_visible/50:.0f} weeks to review them all, ranking is the whole point.")

Visible pages (>=500 impressions_90d): 16,726
Of those, declining: 9,961 (59.6%)

A content team going through ~50 pages/week would need 335 weeks to review them all, ranking is the whole point.


## 3. Quick look at the data (2-3 real numbers)


Three numbers, each one a reason this lane is worth :

1. Decline is the majority outcome, not an edge case. 54.2% of pages in the starter CSV show a downward trend. This is a real prioritization problem, not a rare event.
2. CTR varies by content type. Feedly articles average ~0.43% CTR, keyword articles ~0.25%, comparison articles ~0.13%, nearly a 5× spread. That spread is a candidate signal for the model.
3. One intent-content combination is rare enough to flag. Navigational intent on a keyword article is 0.15% of the dataset (46 pages). Too small to prove malice, but exactly the kind of thin bucket that deserves a human look.

In [5]:
print("NUMBER 1: Decline rate (observed outcome)")
print(f"  Declining pages: {df['is_declining_label'].mean():.1%}")
print()

print("NUMBER 2: CTR spread across content types (visible pages only)")
ctr_by_content = visible.groupby("content_type")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_content.round(3).to_string())
print(f"  Spread: {ctr_by_content.max():.3f} to {ctr_by_content.min():.3f} "
      f"({ctr_by_content.max()/ctr_by_content.min():.1f}x)")
print()

print("NUMBER 3: Rare intent-content combination")
navig_kw = len(df[(df["main_intent"] == "navigational") & (df["content_type"] == "keyword article")])
print(f"  navigational x keyword article: {navig_kw} pages "
      f"({navig_kw/len(df)*100:.2f}% of the dataset)")
print()

print("Does not yet test whether mismatch PREDICTS decline.")


NUMBER 1: Decline rate (observed outcome)
  Declining pages: 54.2%

NUMBER 2: CTR spread across content types (visible pages only)
content_type
feedly article        0.430
keyword article       0.262
comparison article    0.088
  Spread: 0.430 to 0.088 (4.9x)

NUMBER 3: Rare intent-content combination
  navigational x keyword article: 46 pages (0.15% of the dataset)

Does not yet test whether mismatch PREDICTS decline.


## 4. Careful words: what I can and can't claim

What I can say, with the evidence I have so far:
- Observed: in this starter CSV, 54.2% of pages trend down over the trailing 90 days.
- Observed:CTR differs across content types by roughly 5× on visible pages.
- Measured: one intent-content combination (navigational × keyword article) is rare (46 pages).

What I cannot say yet, and will not claim:
- That intent-content mismatch causes decline. This is one snapshot; no intervention was run. The honest form is decision-support: "these pages look worth reviewing first."
- That rare combinations are malicious. Rare is not the same as malicious.
- That any pattern here generalizes beyond this portfolio. It is one dataset, one window.
- Anything about Google's algorithm. I am modeling observed outcomes in this data, not the ranking system.



In [6]:

print("Evidence window of the starter CSV (trailing 90 days per page):")
print(f"  Rows: {len(df):,}")
print(f"  Clients: {df['client_id'].nunique()}")
print(f"  Distinct content types: {df['content_type'].nunique()}")
print(f"  Distinct intent types: {df['main_intent'].nunique()}")



Evidence window of the starter CSV (trailing 90 days per page):
  Rows: 30,000
  Clients: 32
  Distinct content types: 3
  Distinct intent types: 4


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.